# group_mean_imputer.py

In [ ]:
# Example Usage

FEATURE_GROUP_MAP = {
    "annual_inc": "grade",
    "dti": "grade",
    "emp_length_years": "grade",
    "bc_util": "fico_band",
    "revol_util_pct": "fico_band",
    "total_bc_limit": "fico_band",
    "mort_acc": "home_ownership",
    "tot_cur_bal": "home_ownership",
    "total_rev_hi_lim": "fico_band",
    "balance_to_limit_ratio": "fico_band",
    "revolving_balance_ratio": "fico_band",
    "loan_to_income_ratio": "grade",
    "funded_to_income_ratio": "grade",
    "installment_income_ratio": "grade",
}

In [ ]:
from preprocessing.group_mean_imputer import FeatureSpecificGroupMeanImputer

imputer = FeatureSpecificGroupMeanImputer(FEATURE_GROUP_MAP)

X_train = imputer.fit_transform(X_train)
X_valid = imputer.transform(X_valid)
X_test = imputer.transform(X_test)

In [ ]:
# View learned statistics
imputer.summary()

# Group means for a feature
imputer.get_group_means()["annual_inc"]

# Global means
imputer.get_global_means()

## missing_category_imputer.py

In [ ]:
# Example Usage

from preprocessing.missing_category_imputer import MissingCategoryImputer

categorical_features = [
    "grade",
    "sub_grade",
    "purpose",
    "application_type",
    "home_ownership",
    "fico_band"
]

cat_imputer = MissingCategoryImputer(
    categorical_features=categorical_features,
    fill_value="Missing"
)

X_train = cat_imputer.fit_transform(X_train)
X_valid = cat_imputer.transform(X_valid)
X_test = cat_imputer.transform(X_test)

In [ ]:
# View the configuration:

cat_imputer.summary()

Example output:

feature	fill_value
grade	Missing
sub_grade	Missing
purpose	Missing
application_type	Missing
home_ownership	Missing
fico_band	Missing

# optimal_binning.py

In [ ]:
# Example Usage
from preprocessing.optimal_binner import OptimalBinner

binner = OptimalBinner(
    name="annual_inc",
    dtype="numerical",
    max_n_bins=6,
    min_bin_size=0.05,
)

binner.fit(
    X_train["annual_inc"],
    y_train,
)

# WoE values
annual_income_woe = binner.transform(
    X_train["annual_inc"],
    metric="woe",
)

# Event rates
event_rates = binner.transform(
    X_train["annual_inc"],
    metric="event_rate",
)

# Bin indices
indices = binner.transform(
    X_train["annual_inc"],
    metric="indices",
)

# View binning table
binner.get_binning_table()

# Plot
binner.plot()

# Save
binner.save("models/binners/annual_income.joblib")

# Load
loaded = OptimalBinner.load(
    "models/binners/annual_income.joblib"
)


After Improvements

Now every fitted feature has one object containing everything.

Instead of repeatedly writing

In [ ]:
table = binner.get_binning_table()

In [ ]:
# you can simply call

binner.statistics

In [ ]:
# Now all the below are available

stats = binner.statistics

stats.woe

stats.goods

stats.bads

stats.event_rate



In [ ]:
#After adding diagnostic property

diag = binner.diagnostics

diag.iv

diag.n_bins

diag.status

In [ ]:
# After adding properties then anywhere in the project. No table parsing required.

binner.woe_values

binner.goods

binner.bads

binner.event_rates

binner.bin_labels

In [ ]:
# After batch summary

OptimalBinner.diagnostics_report(
    {
        "fico_score": fico_binner,
        "annual_inc": income_binner,
        "dti": dti_binner,
    }
)

## Why This Design Matters

These additions make the rest of the preprocessing library much simpler:

- WOETransformer can access binner.statistics directly instead of parsing binning tables repeatedly.
- IVCalculator can build a global IV summary from binner.iv and binner.diagnostics.
- Scorecard can generate score tables directly from binner.woe_values and binner.bin_labels.
- The diagnostics report provides a ready-made audit artifact for model governance, showing optimization status, IV, and bin counts for every feature.

This creates a cleaner separation of responsibilities: OptimalBinner encapsulates everything about a single feature's binning, while downstream components consume structured objects rather than raw optbinning tables. That makes the entire library easier to test, maintain, and eventually port to Spark when you build your distributed ML pipeline.

# woe_transformer.py

In [ ]:
# Example Usage

import pandas as pd
import numpy as np
from woe_transformer import WOETransformer

# ---------------------------------------------------------
# 1. Create a dummy dataset
# ---------------------------------------------------------
np.random.seed(42)
n_samples = 1000

data = pd.DataFrame({
    "age": np.random.randint(18, 70, n_samples),
    "income": np.random.normal(50000, 15000, n_samples),
    "home_ownership": np.random.choice(["RENT", "OWN", "MORTGAGE"], n_samples),
    "grade": np.random.choice(["A", "B", "C", "D"], n_samples)
})

# Create a dummy binary target (1 = default, 0 = no default)
# Let's make 'grade' and 'income' somewhat predictive
target_prob = np.where(data["grade"].isin(["C", "D"]), 0.4, 0.1)
target_prob += np.where(data["income"] < 40000, 0.2, 0)
y = pd.Series(np.random.binomial(1, np.clip(target_prob, 0, 1)), name="default")

# Split into train and test for realism
X_train, y_train = data.iloc[:800], y.iloc[:800]
X_test, y_test = data.iloc[800:], y.iloc[800:]

# ---------------------------------------------------------
# 2. Instantiate the WOETransformer
# ---------------------------------------------------------
print("Initializing WOETransformer...")
woe_transformer = WOETransformer(
    max_n_bins=5,
    min_bin_size=0.05,
    monotonic_trend="auto"
)

# ---------------------------------------------------------
# 3. Fit and Transform the Training Data
# ---------------------------------------------------------
print("\nFitting and transforming X_train...")
X_train_woe = woe_transformer.fit_transform(X_train, y_train)

print("\nTransformed Training Data (First 5 rows):")
print(X_train_woe.head())

# ---------------------------------------------------------
# 4. Transform the Test Data
# ---------------------------------------------------------
print("\nTransforming X_test (applying learned bins)...")
X_test_woe = woe_transformer.transform(X_test)

# ---------------------------------------------------------
# 5. Review Diagnostics and Summaries
# ---------------------------------------------------------
print("\nInformation Value (IV) Summary Table:")
print(woe_transformer.summary())

print("\nBinning Table for 'income':")
# This returns the detailed bin-by-bin breakdown
income_bins = woe_transformer.get_binning_table("income")
print(income_bins[["Bin", "Count", "Event rate", "WoE", "IV"]])

# ---------------------------------------------------------
# 6. Serialization (Optional)
# ---------------------------------------------------------
# woe_transformer.save("woe_transformer.pkl")
# loaded_transformer = WOETransformer.load("woe_transformer.pkl")

# iv_calculator.py

In [ ]:
#Example Usage

from preprocessing.iv_calculator import IVCalculator

calculator = IVCalculator()

calculator.fit(
    woe.binners_
)

calculator.summary()

In [ ]:
# Feature Selection

selected = calculator.select_features(
    min_iv=0.02,
    max_iv=0.50
)

In [ ]:
# Weak Variables

calculator.weak_features()

In [ ]:
# Possible Leakage

calculator.suspicious_features()

In [ ]:
#IV Plot

calculator.plot()

In [ ]:
# Export to CSV

calculator.export(
    "reports/iv_summary.csv"
)

In [ ]:
# Export to Excel

calculator.export(
    "reports/iv_summary.xlsx"
)

# iv_feature_selector.py

For production use, it should be a proper sklearn transformer so it can be inserted directly into your preprocessing pipeline:

In [ ]:
Pipeline([
    ("imputer", FeatureSpecificGroupMeanImputer(...)),
    ("missing", MissingCategoryImputer(...)),
    ("woe", WOETransformer(...)),
    ("iv", IVFeatureSelector(min_iv=0.02, max_iv=0.50)),
])

In [ ]:
## Example Usage

from preprocessing.iv_feature_selector import IVFeatureSelector

selector = IVFeatureSelector(
    min_iv=0.02,
    max_iv=0.50
)

selector.fit(
    X_train_woe,
    iv_summary=calculator.summary()
)

X_train_selected = selector.transform(X_train_woe)

X_valid_selected = selector.transform(X_valid_woe)

X_test_selected = selector.transform(X_test_woe)

In [ ]:
# Export

selector.export(
    "reports/iv_selection.xlsx"
)

# Summary

selector.summary()

# correlation_filter.py

In [ ]:
# Example Usage

corr_filter = CorrelationFilter(
    threshold=0.70,
    iv_source=calculator,
)

X_train = corr_filter.fit_transform(X_train)

X_valid = corr_filter.transform(X_valid)

X_test = corr_filter.transform(X_test)

In [ ]:
# Summary

corr_filter.summary()

In [ ]:
# Plot

corr_filter.plot()

# vif_selector.py

In [ ]:
# How it fits into the pipeline

# Step 1: WOE transformation
X_woe = woe.transform(X_train)

# Step 2: IV filtering already applied
X_iv = iv_selector.transform(X_woe)

# Step 3: Correlation filtering
X_corr = corr_filter.transform(X_iv)

# Step 4: VIF filtering (final multicollinearity cleanup)
vif_selector = VIFSelector(threshold=5.0)

X_train_final = vif_selector.fit_transform(X_corr)
X_valid_final = vif_selector.transform(X_valid_corr)
X_test_final  = vif_selector.transform(X_test_corr)

# preprocessing_pipeline.py

In [ ]:
# preprocessing
pipe = PreprocessingPipeline()

X_train_ready = pipe.fit_transform(X_train, y_train)
X_valid_ready = pipe.transform(X_valid)
X_test_ready  = pipe.transform(X_test)

# scorecard_scaler.py

In [ ]:
## Example Usage

# model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=200)

model.fit(X_train_ready, y_train)

# scorecard
scaler = ScoreScaler()

scaler.fit(model, pipe.selected_features_)

scores = scaler.calculate_score(X_test_ready)